In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split

from utils import multi_hot_encode

RANDOM_STATE = 36

I0000 00:00:1785656464.049840   12099 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785656464.266760   12099 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785656468.183251   12099 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


# 1. Importing Data

In [2]:
movies = pd.read_csv('./MovieLens/movies.csv')
ratings = pd.read_csv('./MovieLens/ratings.csv')

print(movies.shape, ratings.shape)
print(movies.columns, ratings.columns)

(9742, 3) (100836, 4)
Index(['movieId', 'title', 'genres'], dtype='str') Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')


In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


# 2. Encoding Genres Column

In [5]:
genres = [
    "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir",
    "Horror", "IMAX", "Musical", "Mystery", "Romance",
    "Sci-Fi", "Thriller", "War", "Western"
]
genre_col = movies['genres']

encoded_columns = multi_hot_encode(genres, genre_col)

movies['genres_encoded'] = encoded_columns


# Mergrin encoded Gernes to Ratings DataFrame

In [6]:
ratings = ratings.merge(
    movies[['movieId', 'genres_encoded']], # Columns
    how='left', # Ratings is the main Df
    on='movieId' # Match based on movieId column
)

# Dropping TimeStamp Column

In [7]:
ratings = ratings.drop('timestamp', axis=1)

In [8]:
ratings

,userId,movieId,rating,genres_encoded
0,1,1,4.0,"[0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
1,1,3,4.0,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
2,1,6,4.0,"[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,1,47,5.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
4,1,50,5.0,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
...,...,...,...,...
100831,610,166534,4.0,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, ..."
100832,610,168248,5.0,"[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
100833,610,168250,5.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ..."
100834,610,168252,5.0,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# Train/Validation Splitting Data

In [9]:
train_data , cv_data = train_test_split(ratings, test_size=0.2, random_state=RANDOM_STATE)

print(train_data.shape, cv_data.shape)

(80668, 4) (20168, 4)


# Splitting Columns

In [ ]:
def seperate_data(train_data, cv_data):
    
    X_user_train = train_data['userId'].values
    y_train = train_data['rating'].values
    
    X_movie_train = train_data['genres_encoded'].values
    X_movie_train = np.stack(X_movie_train['genres_encoded'].values)
    
    train_movie_ids = train_data['movieId'].values

    #  - - - - - - - - - - - -- -- - - - ----------

    X_user_cv = cv_data['userId'].values
    X_movie_cv = cv_data['genres_encoded'].values
    X_movie_cv = np.stack(X_movie_cv['genres_encoded'].values)
    
    y_cv = cv_data['rating'].values
    cv_movie_ids = cv_data['movieId'].values
    

array([292, 274, 599, 414, 391])

# Scaling Data

In [ ]:
# scaler_train = StandardScaler()
# print(train_data.dtypes)
# train_scaled = scaler_train.fit_transform(train_data)

# scaler_cv = StandardScaler()

# cv_scaled = scaler_cv.fit_transform(cv_data)